In [ ]:
from typing import Any
from astropy.io import ascii
from snal.helper import AnalysisHelper
import numpy as np
from snal.model import DOMInteractionL17
from lmfit import minimize, Parameters
from astropy.table import vstack
import matplotlib.pyplot as plt
from snal.helper import ABVegaMagnitude
import matplotlib
#%matplotlib inline 
matplotlib.use('Agg')

helper = AnalysisHelper()
path_imsng = './data/SN2021aefx_formatted_Host_dereddening_MW_dereddening.ascii_fixed_width'
path_h22 = './data/Hosseinzadeh2022_formatted_Host_dereddening_MW_dereddening.ascii_fixed_width'

tbl_imsng = ascii.read(path_imsng, format = 'fixed_width')
tbl_h22 = ascii.read(path_h22, format = 'fixed_width')
tbl_h22 = tbl_h22[tbl_h22['observatory'] == 'LasCumbres1m']
tbl_all = vstack([tbl_imsng, tbl_h22])
DM = 31.133854 # From 3.SNcosmo.py

In [ ]:
# # CONVERT VEGA TO AB MAGNITUDE
# from snal.helper import ABVegaMagnitude
# mag = ABVegaMagnitude(tbl_all['mag'], magsys = tbl_all['magsys'], filter_ = tbl_all['filter'])
# tbl_all['mag'] = mag.AB
# tbl_all['magsys'] = 'AB'

In [ ]:
from ezphot.dataobjects import LightCurve
lc_imsng = LightCurve()
lc_imsng.data = tbl_imsng
lc_imsng.plt_params.xlim = [59500, 59730]
lc_imsng.plt_params.ylim = [20, 8]
lc_imsng.plt_params.figure_figsize = (12, 8)
fig, ax, _= lc_imsng.plot(ra = 64.9725, dec= -54.948081, flux_key = 'mag', fluxerr_key = 'e_mag')

In [ ]:
from astropy.table import vstack
lc_all = LightCurve()
lc_all.data = tbl_all
lc_all.plt_params.xlim = [59500, 59730]
lc_all.plt_params.ylim = [20, 8]
lc_all.plt_params.figure_figsize = (12, 8)
lc_all.plot(ra = 64.9725, dec= -54.948081, flux_key = 'mag', fluxerr_key = 'e_mag')

(<Figure size 3600x2400 with 1 Axes>,
 <Axes: title={'center': 'J041953.40-545653.1'}, xlabel='Obsdate [MJD]', ylabel='Magnitude'>,
 <Table length=1060>
        mjd           mag    e_mag  ...      depth_y       label      group     
      float64       float64 float64 ...      float64        str6      str14     
 ------------------ ------- ------- ... ------------------ ------ --------------
 59503.317127222224      --      -- ... 19.096172429919882      g          g|KCT
  59506.29273108796      --      -- ... 18.818972638116815      g          g|KCT
  59507.30140541666      --      -- ...  18.55402208976658      g          g|KCT
   59508.2986994213      --      -- ... 18.638275427578943      g          g|KCT
 59509.296315162035      --      -- ... 18.908664151029797      g          g|KCT
  59510.29526356482      --      -- ... 18.878355733168092      g          g|KCT
  59511.34889185185      --      -- ... 19.045279868114502      g          g|KCT
  59512.34537891204      --      -- .

In [ ]:
def fireball_model(time, amplitude, exptime, alpha):
    dt = np.asarray(time) - exptime
    dt = np.clip(dt, 1e-6, None)  # avoid <=0
    flux = amplitude * (dt**alpha)
    return np.nan_to_num(flux, nan=1e-6, posinf=1e6, neginf=1e-6)

def band_model(params, time, b):
    exptime = params['exptime'].value
    amp     = params[f'amp_{b}'].value
    alpha   = params[f'alpha_{b}'].value
    return fireball_model(time, amp, exptime, alpha)

def residuals(params, x_list, y_list, e_list, band_list):
    res = []
    for i, b in enumerate(band_list):
        mod = band_model(params, x_list[i], b)
        res.append((y_list[i] - mod) / e_list[i])  # unsquared residuals
    return np.concatenate(res)

def get_DEI_spline(model_DEI,
                   exptime_DEI,
                   filterset : str = 'UBVRIugri',
                   smooth : float = 0.05):
    spl_dict = dict()
    for filter_ in filterset:
        model_mag = model_DEI[filter_]
        inf_idx = np.isinf(model_mag)
        mag_DEI = model_mag[~inf_idx]
        phase_DEI = model_DEI['phase'][~inf_idx]
        spl, _ = helper.interpolate_spline(phase_DEI + exptime_DEI, mag_DEI, show = False, smooth = smooth)
        spl_dict[filter_] = spl
    return spl_dict

def fit_both(fit_tbl,
             E_exp,
             M_ej,
             kappa,
             M_dom,
             V_dom,
             f_dom,
             t_delay,
             f_comp,
             
             fit_method = 'leastsq'
             ):

    def chisq_both(params, x_fit, y_fit, e_y_fit, filter_key, model_DEI):
        vals = params.valuesdict()
        exptime_DEI = vals['exptime_DEI']
        exptime_FB = vals['exptime_FB']
        chisq_allfilter = []
        spl_allfilt_DEI = get_DEI_spline(model_DEI = model_DEI, exptime_DEI = exptime_DEI, filterset = filter_key)
        for mjd, obs_flux, obs_fluxerr, filter_ in zip(x_fit, y_fit, e_y_fit, filter_key):
            spl_DEI = spl_allfilt_DEI[filter_]
            fireball_alpha = vals[f'alpha_{filter_}']
            fireball_amplitude = vals[f'amplitude_{filter_}']
            fireball_flux = fireball_model(time = mjd, amplitude = fireball_amplitude, alpha = fireball_alpha, exptime = exptime_FB)
            DEI_flux = helper.mag_to_flux(spl_DEI(mjd)+DM)
            both_flux = fireball_flux + DEI_flux
            chisq_singlefilter = ((obs_flux - both_flux)/obs_fluxerr)
            chisq_allfilter.append(chisq_singlefilter)
        return np.concatenate(chisq_allfilter)

    # Input
    filter_tbls = fit_tbl.group_by('filter').groups
    filter_key = filter_tbls.keys['filter']
    fit_table = {filter_:filter_tbl for filter_, filter_tbl in zip(filter_key, filter_tbls)}
    x_fit = [np.array((fit_table[filter_]['mjd'].tolist())) for filter_ in filter_key]
    y_fit = [np.array((fit_table[filter_]['flux'].tolist())) for filter_ in filter_key]
    e_y_fit = [np.array((fit_table[filter_]['e_flux'].tolist())) for filter_ in filter_key]

    # Parameters
    fit_params_DEI_FB = Parameters()
    fit_params_DEI_FB.add('exptime_DEI', value = 59528.5, min = 59525, max = 59535)
    fit_params_DEI_FB.add('exptime_FB', value = 59528.5, min = 59525, max = 59535)
    for filter_ in filter_key:
        fit_params_DEI_FB.add(f'alpha_{filter_}', value = 2, min = 0, max = 4)
        fit_params_DEI_FB.add(f'amplitude_{filter_}', value = 3000, min = 1, max = 500000)
    
    # Fitting
    t_range = np.arange(0.1, 10, 0.1)
    DOM_model = DOMInteractionL17(E_exp = E_exp, M_ej = M_ej, kappa = kappa, M_dom = M_dom, V_dom = V_dom, f_dom = f_dom, t_delay = t_delay, f_comp = f_comp)
    model_DEI = DOM_model.get_LC(td = t_range, filterset = ''.join(filter_key), search_directory = model_directory, save = False)
    out = minimize(chisq_both, fit_params_DEI_FB, args = (x_fit, y_fit, e_y_fit, filter_key, model_DEI), method = fit_method)
    return out

In [ ]:
model_directory = '/home/hhchoi1022/snal/model/DOM_model'
result_directory = '/home/hhchoi1022/snal/result/SN2021aefx/DOM_fit_result'
fit_filterset : str = 'BVgri'
fit_start_mjd : int = 59529
fit_end_mjd : int = 59538.27207782408 # Half maximum mag MJD

# Construct table for fitting
fit_idx = [filter_ in fit_filterset for filter_ in tbl_all['filter']]
fit_tbl = tbl_all[fit_idx]
fit_tbl = fit_tbl[(fit_tbl['mjd'] > fit_start_mjd)&(fit_tbl['mjd'] < fit_end_mjd)]
fit_tbl.sort('mjd')
fit_tbl['flux'] = helper.mag_to_flux(fit_tbl['mag'])
fit_tbl['e_flux'] = fit_tbl['e_mag']*helper.mag_to_flux(fit_tbl['mag'])*2.303/2.5
fit_tbl['absmag'] = (fit_tbl['mag'] - DM).round(3)

lc_fit = LightCurve()
lc_fit.data = fit_tbl
lc_fit.plt_params.xlim = [fit_start_mjd, fit_end_mjd]
lc_fit.plt_params.ylim = [20, 8]
lc_fit.plt_params.figure_figsize = (6, 8)
lc_fit.plot(ra = 64.9725, dec= -54.948081, flux_key = 'mag', fluxerr_key = 'e_mag')

(<Figure size 1800x2400 with 1 Axes>,
 <Axes: title={'center': 'J041953.40-545653.1'}, xlabel='Obsdate [MJD]', ylabel='Magnitude'>,
 <Table length=155>
        mjd           mag          e_mag         ... label      group     
      float64       float64       float64        ...  str6      str14     
 ------------------ ------- -------------------- ... ------ --------------
  59529.33180266204  17.109 0.036144181671755315 ...  r+1.0       r|RASA36
            59529.8  16.374                 0.02 ... B+-1.5 B|LasCumbres1m
          59529.802  16.314                 0.02 ... B+-1.5 B|LasCumbres1m
          59529.805   16.21                 0.02 ... V+-0.5 V|LasCumbres1m
          59529.807   16.25                 0.02 ... V+-0.5 V|LasCumbres1m
          59529.809  16.094                 0.02 ...      g g|LasCumbres1m
          59529.811  16.124                 0.02 ...      g g|LasCumbres1m
          59529.814   16.19                 0.02 ...  r+1.0 r|LasCumbres1m
          59529.816   1

In [ ]:
import numpy as np
import os
import time
import multiprocessing as mp
from astropy.table import Table
#home_dir = '/data7/yunyi/temp_supernova/Gitrepo/Research/model/DOM_model/' 
range_E_exp = np.round(np.arange(1.0, 1.6, 0.1), 1)  # 10^51 ergs, rounded to 2 decimal places
range_M_ej = np.round([1.5], 1)   # solar mass, rounded to 2 decimal places
range_kappa = np.round([0.05], 2)              # cm^2/g, rounded to 2 decimal places
range_t_delay = np.round(np.logspace(1, 3.5, 30), 0)  # s, rounded to 0 decimal places (integer-like)
range_t_delay = np.round(np.arange(1e1, 1e3, 10), 0)
range_f_comp = np.round([1.5], 1)                    # compress fraction, rounded to 2 decimal places
range_M_dom = np.round(np.logspace(-4, -1, 100), 5)  # solar mass, rounded to 2 decimal places
range_M_dom = np.round(np.arange(0.01, 0.7, 0.02), 2)
range_v_dom = np.int_(np.arange(1e3, 5e3, 200))  # km/s, rounded and converted to integer
range_f_dom = np.round([0.1], 1)  # fraction of DOM mass, rounded to 2 decimal places  # fraction of DOM mass, rounded to 2 decimal placestd = np.arange(0.1, 10, 0.1)
len(range_E_exp) * len(range_M_ej) * len(range_kappa) * len(range_t_delay) * len(range_f_comp) * len(range_M_dom) * len(range_v_dom) * len(range_f_dom)

485100

In [ ]:
def process_combination(args):
    E_exp, M_ej, kappa, t_delay, f_comp, M_dom, V_dom, f_dom, fit_tbl = args
    header_parameters = ['E_exp','M_ej','kappa','t_delay','f_comp','M_dom','V_dom','f_dom']
    header_fitvalues = ['exptime_DEI', 'exptime_FB']
    header_fitconfig = ['success','nfev', 'ndata', 'nvar', 'chisq', 'redchisqr', 'aic', 'bic']
    for filter_ in fit_filterset:
        header_fitvalues.append(f'alpha_{filter_}')
        header_fitvalues.append(f'amplitude_{filter_}')
    tot_header = header_parameters + header_fitvalues + header_fitconfig
    result_tbl = Table(names = tot_header)
    result_tbl.add_row(vals = np.zeros(len(result_tbl.colnames)))
    if os.path.exists(f'{result_directory}/kappa{kappa}/E{E_exp}/{E_exp}_{M_ej}_{kappa}_{t_delay}_{f_comp}_{M_dom}_{V_dom}_{f_dom}.fit'):
        return
    try:
        result = fit_both(
                          fit_tbl=fit_tbl,
                          E_exp=E_exp,
                          M_ej=M_ej,
                          kappa=kappa,
                          M_dom=M_dom,
                          V_dom=V_dom,
                          f_dom=f_dom,
                          t_delay=t_delay,
                          f_comp=f_comp,
                          fit_method='leastsq'
                          )
        data_parameters = dict(E_exp=E_exp, M_ej=M_ej, kappa=kappa, t_delay=t_delay, f_comp=f_comp, M_dom=M_dom, V_dom=V_dom, f_dom=f_dom)
        data_fitvalues = result.params.valuesdict()
        data_fitconfig = dict(success=result.success, nfev=result.nfev, ndata=result.ndata, nvar=result.nvarys, chisq=result.chisqr, redchisqr=result.redchi, aic=result.aic, bic=result.bic)
        all_data = {**data_parameters, **data_fitvalues, **data_fitconfig}
        all_values = [all_data[colname] for colname in result_tbl.columns]
        result_tbl.add_row(vals=all_values)
    except:
        data_parameters = dict(E_exp=E_exp, M_ej=M_ej, kappa=kappa, t_delay=t_delay, f_comp=f_comp, M_dom=M_dom, V_dom=V_dom, f_dom=f_dom)
        data_fitvalues = {value: 99999 for value in header_fitvalues}
        data_fitconfig = dict(success=False, nfev=99999, ndata=99999, nvar=99999, chisq=99999, redchisqr=99999, aic=99999, bic=99999)
        all_data = {**data_parameters, **data_fitvalues, **data_fitconfig}
        all_values = [all_data[colname] for colname in result_tbl.columns]
        result_tbl.add_row(vals=all_values)
    os.makedirs(f'{result_directory}/kappa{kappa}/E{E_exp}', exist_ok = True)
    result_tbl.remove_row(index = 0)
    result_tbl.write(f'{result_directory}/kappa{kappa}/E{E_exp}/{E_exp}_{M_ej}_{kappa}_{t_delay}_{f_comp}_{M_dom}_{V_dom}_{f_dom}.fit', format='ascii.fixed_width', overwrite=True)

In [ ]:
# Prepare the list of all combinations of parameters
all_combinations = [(E_exp, M_ej, kappa, t_delay, f_comp, M_dom, V_dom, f_dom, fit_tbl)
                    for E_exp in range_E_exp
                    for M_ej in range_M_ej
                    for kappa in range_kappa
                    for t_delay in range_t_delay
                    for f_comp in range_f_comp
                    for M_dom in range_M_dom
                    for V_dom in range_v_dom
                    for f_dom in range_f_dom]

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from astropy.table import vstack, Table
from astropy.io import ascii
from tqdm import tqdm
import glob

def construct_file_format(E_exp, M_ej, kappa, t_delay, f_comp, M_dom, V_dom, f_dom):
    return f'{result_directory}/kappa{kappa}/E{E_exp}/{E_exp}_{M_ej}_{kappa}_{t_delay}_{f_comp}_{M_dom}_{V_dom}_{f_dom}.fit'

def read_one_file(file_):
    """Function executed in each process."""
    try:
        tbl = ascii.read(file_, format='fixed_width')
        return tbl
    except:
        return None

# Collect file list
files = [construct_file_format(E_exp, M_ej, kappa, t_delay, f_comp, M_dom, V_dom, f_dom)
         for E_exp in range_E_exp
         for M_ej in range_M_ej
         for kappa in range_kappa
         for t_delay in range_t_delay
         for f_comp in range_f_comp
         for M_dom in range_M_dom
         for V_dom in range_v_dom
         for f_dom in range_f_dom]

#files = glob.glob(f'{result_directory}/*/*/*.fit')

In [ ]:
# Read in parallel
tables = []
with ProcessPoolExecutor(60) as executor:
    for tbl in tqdm(executor.map(read_one_file, files), total=len(files), desc="Reading files"):
        if tbl is not None:
            tables.append(tbl)

Reading files: 100%|██████████| 485100/485100 [20:44<00:00, 389.69it/s] 


In [ ]:
result_tbl

NameError: name 'result_tbl' is not defined

In [ ]:
tables

[<Table length=1>
  E_exp    M_ej   kappa  ...       aic               bic       
 float64 float64 float64 ...     float64           float64     
 ------- ------- ------- ... ---------------- -----------------
     1.0     1.5    0.05 ... 593.200772886234 629.7218742892649,
 <Table length=1>
  E_exp    M_ej   kappa  ...        aic               bic       
 float64 float64 float64 ...      float64           float64     
 ------- ------- ------- ... ----------------- -----------------
     1.0     1.5    0.05 ... 635.8682607742539 672.3893621772849,
 <Table length=1>
  E_exp    M_ej   kappa  ...        aic               bic       
 float64 float64 float64 ...      float64           float64     
 ------- ------- ------- ... ----------------- -----------------
     1.0     1.5    0.05 ... 593.1328157922502 629.6539171952811,
 <Table length=1>
  E_exp    M_ej   kappa  ...     redchisqr           aic               bic       
 float64 float64 float64 ...      float64          float64         

In [ ]:
# Stack all tables
result_tbl = vstack(tables)

# Sort and save
result_tbl.sort('redchisqr')
result_tbl.write(f'{result_directory}/DOM_fit_result.fit',
                 format='ascii.fixed_width',
                 overwrite=True)

In [ ]:
result_tbl

E_exp,M_ej,kappa,t_delay,f_comp,M_dom,V_dom,f_dom,exptime_DEI,exptime_FB,alpha_B,amplitude_B,alpha_V,amplitude_V,alpha_g,amplitude_g,alpha_r,amplitude_r,alpha_i,amplitude_i,success,nfev,ndata,nvar,chisq,redchisqr,aic,bic
float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
1.6,1.5,0.05,190.0,1.5,0.01,4800.0,0.1,59527.79924536566,59529.21955649423,2.4497399629396033,625.851157867975,1.8492450906338183,1717.746415828621,2.1819227099266634,1151.251446497093,1.8504834243380355,1728.6102554196366,2.0819527291182776,696.1729192643201,1.0,170.0,155.0,12.0,2205.8939051996235,15.425831504892471,435.59676313893914,472.1178645419701
1.6,1.5,0.05,200.0,1.5,0.01,4600.0,0.1,59527.785570754655,59529.24539260115,2.440672323597816,642.8823118865115,1.841847446373048,1754.1978805067681,2.173294326330314,1180.2801626756507,1.8425386823239127,1766.8839057947102,2.075622246304364,709.9966341342398,1.0,144.0,155.0,12.0,2208.5826016645606,15.44463357807385,435.78557284398534,472.3066742470163
1.6,1.5,0.05,200.0,1.5,0.01,4400.0,0.1,59527.79952156924,59529.219320039956,2.450109758695661,625.3183248901446,1.849629372178288,1716.2063464643877,2.1822819243165243,1150.3019722447048,1.8509199923009967,1726.866269596234,2.082710132791113,694.9532689853881,1.0,170.0,155.0,12.0,2209.6158257480174,15.451858921314807,435.8580683366557,472.3791697396867
1.6,1.5,0.05,190.0,1.5,0.01,4600.0,0.1,59527.81234620925,59529.19493675947,2.4588061569042656,609.489158158381,1.8567253913883686,1682.1147518441373,2.1905413985701494,1123.2957676244996,1.858543127699157,1691.0885900251162,2.088797314075344,682.0088559212385,1.0,209.0,155.0,12.0,2211.0347405430607,15.461781402399026,435.95757034319814,472.4786717462291
1.6,1.5,0.05,210.0,1.5,0.01,4200.0,0.1,59527.78963686478,59529.238677700494,2.4433425888447875,637.9906835671545,1.8441118908677117,1743.3195395526125,2.1758411946773486,1171.9071872896964,1.844993065369562,1755.3270764820675,2.0780322819004606,705.1601524342167,1.0,157.0,155.0,12.0,2211.2758829652157,15.463467713043466,435.9744742102727,472.4955756133037
1.6,1.5,0.05,200.0,1.5,0.01,4800.0,0.1,59527.77326471655,59529.26871743322,2.432397118618697,658.7821610609734,1.8350160226492458,1788.4143194371861,2.1653922588168624,1207.4083506034265,1.8351853114925822,1802.9384146513592,2.069394664722476,723.557944556774,1.0,144.0,155.0,12.0,2212.03673672959,15.468788368738391,436.0277972940249,472.54889869705585
1.6,1.5,0.05,210.0,1.5,0.01,4000.0,0.1,59527.80666147468,59529.20776176957,2.4545863756279096,617.3056668041688,1.8533664670258088,1698.5454996384055,2.186536787943214,1136.601862287099,1.854955295840453,1708.2018304088313,2.086418172848257,687.5082494060626,1.0,183.0,155.0,12.0,2214.048878742621,15.482859291906442,436.1687263499,472.68982775293097
1.6,1.5,0.05,220.0,1.5,0.01,3800.0,0.1,59527.80050485992,59529.22009537673,2.450267846828195,625.2559774930554,1.8498613797693693,1715.5467498298276,2.182435772580702,1150.1516953005555,1.8511934362960734,1726.021061448452,2.0835078027138114,693.8115345655698,1.0,170.0,155.0,12.0,2214.2777908492535,15.484460075868906,436.184751083036,472.705852486067
1.6,1.5,0.05,220.0,1.5,0.01,4000.0,0.1,59527.78223320524,59529.253177435916,2.4383716689357904,647.5156323261183,1.8400726494946533,1763.5509767807755,2.1711062167038437,1188.1282690925918,1.840660284273769,1776.545131051017,2.074672531679203,712.6760100308989,1.0,131.0,155.0,12.0,2214.5342581136083,15.486253553242015,436.202702816178,472.72380421920894
